In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
print(PROJECT_ROOT)

/Users/florianb/Downloads/ai-customer-insights-engine


In [2]:
from src.rag import rag_chain as rag_chain_module
from src.rag import retriever as retriever_module

from config import config

if config.OPENAI_API_KEY is None:
    raise ValueError("OPENAI_API_KEY is missing.")

In [3]:
import importlib

importlib.reload(config)
importlib.reload(rag_chain_module)
importlib.reload(retriever_module)

<module 'src.rag.retriever' from '/Users/florianb/Downloads/ai-customer-insights-engine/src/rag/retriever.py'>

In [4]:
print(f"LLM_MODEL = {config.LLM_MODEL}")
print(f"MAX_COMPLETION_TOKENS = {config.MAX_COMPLETION_TOKENS}")
print(f"RETRIEVER_K = {config.RETRIEVER_K}")
print(f"RERANKER_TOP_N = {config.RERANKER_TOP_N}")

LLM_MODEL = gpt-4o-mini
MAX_COMPLETION_TOKENS = 300
RETRIEVER_K = 5
RERANKER_TOP_N = 5


### Validation de `create_rag_chain()`

In [5]:
retriever = retriever_module.create_retriever(
    model_name=config.HUGGINGFACE_EMBEDDING_MODEL,
    use_reranker=True,
    retriever_k=20,
    reranker_top_n=5,
)

In [6]:
rag_chain = rag_chain_module.create_rag_chain(
    llm_model="gpt-4o-mini",
    max_completion_tokens=300,
    retriever=retriever,
)

In [7]:
type(rag_chain)

langchain_core.runnables.base.RunnableSequence

In [8]:
response = rag_chain.invoke("Quels sont les problèmes avec le service client ?")

In [10]:
response["question"]

'Quels sont les problèmes avec le service client ?'

In [11]:
response["context"]

[Document(id='867d6431-52d8-465f-9dae-9e5887fc8349', metadata={'publication_date': '2024-11-06T10:22:44+01:00', 'experience_date': '2024-07-04T00:00:00', 'dataset': 'processed_reviews', 'review_id': 30215, 'rating': 3, 'bank': 'hellobank', 'title': 'Deçu'}, page_content='Service client pas au top.'),
 Document(id='1ccab572-946b-4b25-a9af-b345c469f2fc', metadata={'publication_date': '2023-03-16T16:48:47+01:00', 'dataset': 'processed_reviews', 'review_id': 17753, 'experience_date': '2023-03-08T00:00:00', 'title': 'Client abandonné', 'bank': 'fortuneo', 'rating': 1}, page_content="Le service clientele est une catastrophe en terme de qualité et d'information fournie erronée, et ne répond jamais aux réclamations, demandes d'information.... Merci pour vos excuses, je les accepte. En revanche les conséquences de cette erreur sont lourdes (360€/an) sans médiation, compensation ou geste commercial. Il est dommage d'obtenir des réponses en 24h sur une plate-forme d'avis client et de rester sans 

In [12]:
response["answer"].pretty_print()

================================== Ai Message ==================================

Les problèmes avec le service client incluent :

1. **Qualité et information erronée** : Le service client est décrit comme une "catastrophe" en termes de qualité et fournit des informations incorrectes.
2. **Absence de réponse** : Les réclamations et demandes d'information ne reçoivent jamais de réponse.
3. **Longs temps d'attente** : Les clients signalent des temps d'attente de plus de 20 minutes, avec des communications qui se coupent.
4. **Complexité des processus** : Ajouter un nouveau bénéficiaire est décrit comme un "parcours de combattant", et même fermer un compte est jugé compliqué.
5. **Incompétence** : Le service client est perçu comme incompétent, ce qui entraîne une perte de temps pour les clients. 

Ces problèmes ont conduit certains clients à envisager de clôturer leur compte et à chercher des alternatives.
